# Renderizar Motion Graphics (HyperFrames) en Google Colab

Este notebook instala Node.js, FFmpeg y Chromium en Colab, sube tu proyecto HyperFrames (zip) y lo renderiza a MP4.

**Pasos:**
1. Ejecuta las celdas en orden.
2. Cuando pida subir archivo, sube un `.zip` de tu carpeta de proyecto (debe contener `index.html`).
3. Descarga el `output.mp4` resultante.

In [ ]:
# 1) Instalar Node.js 22, FFmpeg y Chromium
!apt-get update -qq
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
!apt-get install -y -qq nodejs ffmpeg chromium-browser
!node --version && npm --version && ffmpeg -version | head -1 && which chromium-browser

In [ ]:
# 2) Configurar rutas para HyperFrames
import os
os.environ['CHROME_PATH'] = '/usr/bin/chromium-browser'
os.environ['HYPERFRAMES_FFMPEG_PATH'] = '/usr/bin/ffmpeg'
print('CHROME_PATH', os.environ['CHROME_PATH'])
print('FFMPEG', os.environ['HYPERFRAMES_FFMPEG_PATH'])

In [ ]:
# 3) Subir el proyecto (zip)
from google.colab import files
uploaded = files.upload()  # sube tu proyecto.zip
print('Subido:', list(uploaded.keys()))

In [ ]:
# 4) Descomprimir
import zipfile, glob, os
zip_name = list(uploaded.keys())[0]
os.makedirs('/content/project', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/project')
# Si el zip tiene una carpeta raíz, mover contenido
entries = os.listdir('/content/project')
if len(entries) == 1 and os.path.isdir(os.path.join('/content/project', entries[0])):
    inner = os.path.join('/content/project', entries[0])
    for f in os.listdir(inner):
        os.rename(os.path.join(inner, f), os.path.join('/content/project', f))
    os.rmdir(inner)
print('Contenido:', os.listdir('/content/project'))

In [ ]:
# 5) Verificar proyecto (check)
!cd /content/project && npx --yes hyperframes check --json 2>&1 | tail -40

In [ ]:
# 6) Renderizar a MP4
!cd /content/project && npx --yes hyperframes render --output /content/output.mp4 2>&1 | tail -40

In [ ]:
# 7) Descargar el video
from google.colab import files
files.download('/content/output.mp4')